# Week 1 — Day 6: TF-IDF + baseline classifier

**Builds on your load/clean step** (`day5_load_clean.ipynb`): same functions are inlined so this notebook runs end-to-end.

**Today:** sparse **TF-IDF** features → **LinearSVC** (strong linear baseline on 50K reviews).

Run from project root with `aclImdb/` present.

In [1]:
import os
import re
import time

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from tqdm.auto import tqdm


def load_data(path):
    texts, labels = [], []
    for label in ["pos", "neg"]:
        folder = os.path.join(path, label)
        files = [f for f in os.listdir(folder) if f.endswith(".txt")]
        for file in tqdm(files, desc=f"{os.path.basename(path)}/{label}"):
            fp = os.path.join(folder, file)
            with open(fp, encoding="utf-8") as f:
                texts.append(f.read())
                labels.append(1 if label == "pos" else 0)
    return texts, labels


def clean_text(text):
    text = text.lower()
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^a-zA-Z]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [2]:
DATA_ROOT = "aclImdb"

train_texts, train_labels = load_data(os.path.join(DATA_ROOT, "train"))
test_texts, test_labels = load_data(os.path.join(DATA_ROOT, "test"))

train_texts = [clean_text(t) for t in tqdm(train_texts, desc="clean train")]
test_texts = [clean_text(t) for t in tqdm(test_texts, desc="clean test")]

print(len(train_texts), len(test_texts), set(train_labels))

train/pos:   0%|          | 0/12500 [00:00<?, ?it/s]

train/neg:   0%|          | 0/12500 [00:00<?, ?it/s]

test/pos:   0%|          | 0/12500 [00:00<?, ?it/s]

test/neg:   0%|          | 0/12500 [00:00<?, ?it/s]

clean train:   0%|          | 0/25000 [00:00<?, ?it/s]

clean test:   0%|          | 0/25000 [00:00<?, ?it/s]

25000 25000 {0, 1}


### Model: TF-IDF (sparse) + linear SVM

- **Unigrams + bigrams** capture short phrases; `max_features` caps vocabulary size for speed/memory.
- **`sublinear_tf=True`** uses log-scaled term frequency (common for text).
- **LinearSVC** fits well on high-dimensional sparse inputs.

In [3]:
model = Pipeline(
    [
        (
            "tfidf",
            TfidfVectorizer(
                ngram_range=(1, 2),
                min_df=3,
                max_features=80_000,
                sublinear_tf=True,
            ),
        ),
        ("clf", LinearSVC(random_state=42)),
    ]
)

t0 = time.perf_counter()
model.fit(train_texts, train_labels)
fit_s = time.perf_counter() - t0
print(f"Fit time: {fit_s:.1f}s")

y_pred = model.predict(test_texts)
acc = accuracy_score(test_labels, y_pred)
print(f"Test accuracy: {acc:.4f}")
print()
print(classification_report(test_labels, y_pred, digits=4))
print("Confusion matrix [rows=true 0,1 | cols=pred 0,1]:")
print(confusion_matrix(test_labels, y_pred))

Fit time: 8.7s
Test accuracy: 0.9007

              precision    recall  f1-score   support

           0     0.8970    0.9054    0.9012     12500
           1     0.9045    0.8961    0.9003     12500

    accuracy                         0.9007     25000
   macro avg     0.9008    0.9007    0.9007     25000
weighted avg     0.9008    0.9007    0.9007     25000

Confusion matrix [rows=true 0,1 | cols=pred 0,1]:
[[11317  1183]
 [ 1299 11201]]
